# EDA

Цель - изучить структуру обучающей выборки и benchmark-данных и определить признаки, которые могут быть полезны для кандидатогенерации

В ходе анализа проверяются:
- структура и качество данных
- распределение поисковых запросов и обьявлений
- пересечение train и benchmark
- количество релевантных обьявлений для запросов
- связ категории и локации запроса с выбранным объявлением
- влияние полного поискового контекста

Результаты EDA будут использованы при построении локальной валидации и выборе методов retrieval.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Загрузка данных
В задаче используются три набора данных:

- `train.parquet` - положительные взаимодействия «поисковый запрос - выбранное объявление»;
- `benchmark_queries.parquet` - запросы, для которых необходимо сформировать кандидатов;
- `benchmark_items.parquet` - корпус объявлений, среди которых выполняется поиск.

Сначала загрузим данные и проверим их размеры и структуру

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

DATA_DIR = Path("../data/raw")

train = pd.read_parquet(DATA_DIR / "train.parquet")
queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

In [5]:
# проверяем
print("Train:", train.shape)
print("Benchmark queries:", queries.shape)
print("Benchmark items:", items.shape)

Train: (497673, 19)
Benchmark queries: (2452, 6)
Benchmark items: (189212, 14)


## Общая структура данных
Рассматриваю доступные признаки, типы данных, количество уникальных значений и долю пропусков. Это позволяет понять, какие признаки доступны одновременно в train и benchmark и могут использоваться при построении retrieval-системы

In [6]:
# посмотрим структуру
display(train.head(3))
display(queries.head(3))
display(items.head(3))

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимости за услугу Работаете с юрлицами и ИП Опыт работы 10 лет и больше Гарантия на вы...",e8b685dffe1a408e,"Скупаю практически любую современную новую и б/у технику(обязательно рабочую), до 80% от рыночной стоимости.\nВсе предложения присылайте в личные ...",114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,"Вид услуги Место оказания услуг Нижний Новгород, Советский район, жилой комплекс Новая Кузнечиха Тип стоимости за услугу Начальная цена Работаете ...",92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля в Нижнем Новгороде и области.\n\n🫴 Помогу подобрать оптимальный вариант автомобиля под ваш бюджет.\...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург, садоводческое некоммерческое товарищество Веретено-2, Цветочная ул., 146 Тип ус...",624846856ce81d69,"В ритме современной жизни так сложно найти момент, чтобы остановиться: выключить телефон, отложить дела и подарить себе и своим близким настоящее...",114


,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114


,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на дом,57.0,5.0,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от 34200 График работы до 828...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\n● Диагностика устройств;\n\n● Чистка от пыли и замена термопасты;\n\n● Проверка работоспособности в...",114
1,Обучение ребенка чтению,7.0,5.0,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское развитие, логопеды Тип стоимости за урок Начальная цена Г...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ, детский психолог/нейропсихолог, рада пригласить вас и ваших детей на захватывающее и увлекательное ...",114
2,Афрокудри 5+,14.0,5.0,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания услуг городской округ Иваново, Фрунзенский район Тип услуги Услуги парикмахера Тип стоимости за услугу...",3f6ae81704565b9c,"ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный вариант замены хим завивки, если хочется поменять временно свой образ.\n\nДля взрослых и детей! \n\n...",114


In [7]:
print("train")
print(train.columns.tolist())

print("\nqueries")
print(queries.columns.tolist())

print("\nitems")
print(items.columns.tolist())

train
['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category', 'item_title_raw', 'item_rating_reviews_count', 'item_rating', 'item_price', 'item_microcat_id', 'item_longitude', 'item_location_id', 'item_latitude', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_infm_params_text', 'item_id', 'item_description_raw', 'item_category_id']

queries
['query_id', 'search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']

items
['item_title_raw', 'item_rating_reviews_count', 'item_rating', 'item_price', 'item_microcat_id', 'item_longitude', 'item_location_id', 'item_latitude', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_infm_params_text', 'item_id', 'item_description_raw', 'item_category_id']


In [8]:
# проверяем типы данных и пропуски
def dataset_info(df, name):
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=False)
    })

    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    return info


display(dataset_info(train, "TRAIN"))
display(dataset_info(queries, "BENCHMARK QUERIES"))
display(dataset_info(items, "BENCHMARK ITEMS"))

TRAIN: 497,673 rows × 19 columns


,dtype,missing,missing_%,n_unique
search_query,object,0,0.00,74529
search_location_id,int64,0,0.00,2782
search_is_delivery_search,int32,0,0.00,2
search_infm_params_text,object,0,0.00,3724
search_category,int64,0,0.00,4
item_title_raw,object,0,0.00,210029
item_rating_reviews_count,float64,19211,3.86,1373
item_rating,float64,28232,5.67,8833
item_price,object,0,0.00,2992
item_microcat_id,int64,0,0.00,212


BENCHMARK QUERIES: 2,452 rows × 6 columns


,dtype,missing,missing_%,n_unique
query_id,object,0,0.0,2452
search_query,object,0,0.0,2452
search_location_id,int64,0,0.0,273
search_is_delivery_search,int32,0,0.0,1
search_infm_params_text,object,0,0.0,132
search_category,int64,0,0.0,2


BENCHMARK ITEMS: 189,212 rows × 14 columns


,dtype,missing,missing_%,n_unique
item_title_raw,object,0,0.00,119952
item_rating_reviews_count,float64,11615,6.14,1171
item_rating,float64,17631,9.32,6008
item_price,object,0,0.00,2949
item_microcat_id,int64,0,0.00,752
item_longitude,object,1,0.00,120005
item_location_id,int64,0,0.00,2877
item_latitude,object,1,0.00,119779
item_is_phone_hidden,bool,0,0.00,2
item_is_message_forbidden,bool,0,0.00,2


В данных практически отсутствуют пропуски в основных текстовых и категориальных признаках. Пропущенные значения встречаются преимущественно в рейтингах и координатах объявлений.

## Пересечение поисковых запросов train и benchmark
Проверяю, встречались ли тексты benchmark-запросов в обучающей выборке. Если запрос уже встречался в train, исторические взаимодействия «запрос - объявление» потенциально можно использовать как дополнительный источник кандидатов.

In [9]:
# Пересекаются ли benchmark-запросы с train?
train_queries = set(train["search_query"].str.lower().str.strip())
benchmark_queries = set(queries["search_query"].str.lower().str.strip())

common_queries = train_queries & benchmark_queries

print(f"Unique train queries: {len(train_queries):,}")
print(f"Unique benchmark queries: {len(benchmark_queries):,}")
print(f"Exact overlap: {len(common_queries):,}")
print(
    f"Benchmark overlap: "
    f"{len(common_queries) / len(benchmark_queries):.2%}"
)

Unique train queries: 74,529
Unique benchmark queries: 2,452
Exact overlap: 907
Benchmark overlap: 36.99%


Около 37% benchmark-запросов в точности встречаются в обучающей выборке. Следовательно, исторические взаимодействия могут быть полезным retrieval-сигналом для части запросов, однако не могут быть единственным методом поиска

In [10]:
list(sorted(common_queries))[:30]

['3 д визуализация',
 '3d визуализация',
 '3d моделирование ювелирных изделий',
 '3d номера',
 '3d сканер',
 '3д сканирование 3d моделирование',
 'arduino',
 'bbl фотоомоложение',
 'lpg массаж',
 'mercedes ключи изготовление',
 'sony playstation 5 чистка',
 'uv dtf наклейки',
 'youtube на телевизоре',
 'авто разбор',
 'автобус в аренду',
 'автовоз перевозка авто',
 'адвокат по земельным вопросам',
 'аксиальный пресс',
 'акт освидетельствования скрытых работ',
 'акт проверки дымоходов и вентканалов',
 'акты проверки дымохода',
 'алмазное бурение без пыли',
 'анализ воды',
 'английский язык разговорный',
 'аниматор на детский праздник в садик',
 'аниматор робот',
 'аниматор шоу мыльных пузырей',
 'аренда авто свадьба',
 'аренда грузовика без водителя',
 'аренда грузового автомобиля с водителем']

##  Пересечение объявлений train и benchmark
Какая часть объявлений benchmark-корпуса уже встречалась среди положительных взаимодействий в train. Такое пересечение позволяет использовать известные взаимодействия с объявлениями как дополнительный сигнал релевантности.

In [11]:
# Объявления benchmark встречались в train?
train_item_ids = set(train["item_id"])
benchmark_item_ids = set(items["item_id"])

common_items = train_item_ids & benchmark_item_ids

print(f"Unique train items: {len(train_item_ids):,}")
print(f"Benchmark items: {len(benchmark_item_ids):,}")
print(f"Items in both: {len(common_items):,}")
print(
    f"Benchmark item overlap: "
    f"{len(common_items) / len(benchmark_item_ids):.2%}"
)

Unique train items: 344,825
Benchmark items: 189,212
Items in both: 18,142
Benchmark item overlap: 9.59%


In [12]:
train_benchmark_items = train[
    train["item_id"].isin(benchmark_item_ids)
]

print("Train interactions involving benchmark items:")
print(f"{len(train_benchmark_items):,}")

display(
    train_benchmark_items[
        ["search_query", "item_id", "item_title_raw"]
    ].head(20)
)

Train interactions involving benchmark items:
33,010


,search_query,item_id,item_title_raw
1,автоподбор,92e1b0446f827b59,Автоподбор Разовый осмотр автомобиля
22,наращивание ресницы,94d689f99943b10d,Наращивание ресниц / Модель
41,маникюр,d3d3e7c283abbbbe,Маникюр Педикюр Наращивание Юго-Запад
45,юрист развод,16757dc134a51ffb,"Семейный юрист: развод, алименты, раздел имущества"
57,услуги эвакуатора,4a3c48aef4ca619d,Эвакуатор. Услуги эвакуатора
67,вывоз мусора,f2bed16df7d7c5e2,Вывоз мусора/переезд/газель/грузчики
124,монтаж межкомнатных дверей,7b25e60bb7513365,Установка межкомнатных дверей в Москве и мо
129,вывоз бытовой техники бесплатно,5f890ef88a6a919a,Вывезу неисправную бытовую технику
198,вывоз мусора,9206b8f65095c9ab,Вывоз мусора ломовоз
214,вскрытие замков,1d507201b6a68916,Вскрытие замков / вскрыть замок / вскрытие авто


В train встречается около 9,6% объявлений benchmark-корпуса. Пересечение относительно небольшое, поэтому исторические взаимодействия следует использовать как дополнительный, а не основной retrieval-механизм

## Количество релевантных объявлений на запрос
Сколько различных объявлений пользователи выбирали для одного текста поискового запроса? Это позволяет оценить сложность retrieval-задачи и понять распределение количества положительных примеров.

In [13]:
# Насколько много релевантных айтемов приходится на запрос?
items_per_query = (
    train.groupby("search_query")["item_id"]
    .nunique()
)

display(items_per_query.describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
))

count    74529.000000
mean         5.974332
std         50.302240
min          1.000000
50%          1.000000
75%          3.000000
90%          7.000000
95%         15.000000
99%         78.000000
max       5474.000000
Name: item_id, dtype: float64

In [14]:
print("Queries with 1 relevant item:",
      (items_per_query == 1).mean())

print("Queries with <= 2 relevant items:",
      (items_per_query <= 2).mean())

print("Queries with <= 5 relevant items:",
      (items_per_query <= 5).mean())

Queries with 1 relevant item: 0.6057776167666278
Queries with <= 2 relevant items: 0.748057803002858
Queries with <= 5 relevant items: 0.8753773698828644


In [15]:
display(
    items_per_query
    .sort_values(ascending=False)
    .head(20)
)

search_query
маникюр                    5474
наращивание ресниц         4189
массаж                     3830
педикюр                    2867
установка кондиционеров    2718
сантехник                  2576
электрик                   2451
вывоз мусора               2349
эвакуатор                  2236
покос травы                1961
манипулятор                1757
наращивание ногтей         1757
автоэлектрик               1746
грузоперевозки газель      1624
вспашка земли              1580
натяжные потолки           1466
грузоперевозки             1350
торты на заказ             1312
грузчики                   1277
поклейка обоев             1248
Name: item_id, dtype: int64

Большинство запросов имеют небольшое число наблюдаемых релевантных объявлений: для 60,6% текстов запросов наблюдается только одно объявление, а для 87,5% не более пяти. При этом популярные общие запросы, например «маникюр», «массаж» или «педикюр», имеют тысячи различных положительных взаимодействий

## Анализ категорий

Проверяю связь между категорией поискового запроса (search_category) и категорией выбранного объявления (item_category_id). Если между ними существует устойчивое соответствие, категорию потенциально можно использовать для фильтрации или дополнительного scoring кандидатов

In [16]:
# Проверяем category
category_cross = pd.crosstab(
    train["search_category"],
    train["item_category_id"],
    normalize="index"
)

display(category_cross.round(3))

item_category_id,10,19,27,28,40,81,112,114
search_category,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
81,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
114,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [17]:
display(
    pd.crosstab(
        train["search_category"],
        train["item_category_id"]
    )
)

item_category_id,10,19,27,28,40,81,112,114
search_category,,,,,,,,
0,0,0,0,0,0,0,0,34
10,0,0,0,0,0,0,0,2
81,0,0,0,0,0,0,0,2
114,1,2,4,2,2,2,2,497620


Практически все положительные взаимодействия train относятся к item_category_id = 114`. При этом около 99% benchmark-корпуса также уже относится к категории 114. Поэтому категория хорошо характеризует область поиска, но практически не сокращает пространство кандидатов в benchmark.

## Анализ локации
Для услуг географическая близость потенциально является важным фактором релевантности. Как часто search_location_id поискового запроса совпадает с item_location_id выбранного объявления

In [18]:
# Проверяем location
same_location = (
    train["search_location_id"]
    == train["item_location_id"]
)

print(f"Exact same location: {same_location.mean():.2%}")

Exact same location: 83.10%


In [19]:
display(
    train.loc[
        ~same_location,
        [
            "search_query",
            "search_location_id",
            "item_location_id",
            "item_title_raw"
        ]
    ].head(20)
)

,search_query,search_location_id,item_location_id,item_title_raw
3,изготовление госномера на авто,634670,633570,"Изготовление дубликатов авто номеров, гос номеров"
5,сборка мебели,639550,637770,Сборка мебели
6,наращивание ногтей,107620,637640,Мастер маникюр и педикюра
19,пополнение alipay,621540,637640,"Пополнение Alipay, Wechat. Быстрый обмен"
20,хромирование,632660,633540,"Хромирование,нанесение хрома,ремонт отражателей"
26,липолитики,107620,637640,Липолитики Стройность Био гель Похудение
29,авто под выкуп,629990,630020,Авто под выкупа
33,муж на час,623845,653271,Услуги Мастера на час / Электрика / Сантехника / Качество / Выезд по всему городу / Гарантия
39,кузовные работы,632660,633540,Ремонт кузова и покраска
61,ремонт кондиционеров igc,637640,637641,Установка и продажа кондиционеров Зеленоград


В 83,1% положительных взаимодействий локация запроса совпадает с локацией выбранного объявления. Это делает локацию сильным сигналом релевантности. Однако использовать её как жёсткий фильтр нельзя, поскольку примерно в 17% положительных примеров локации различаются. Поэтому далее локацию целесообразно рассматривать как дополнительный фактор при scoring кандидатов

## Выводы предварительные
1. 37% benchmark-запросов встречались в train буквально. Значит, для части benchmark можно использовать историю взаимодействий как дополнительный retrieval-канал. Но нельзя просто возвращать старые clicked items: только 9.59% benchmark-корпуса пересекается с train items, и конкретное старое объявление может вообще отсутствовать в benchmark
2. Train и benchmark corpus пересекаются на 18 142 объявления. Причём для них есть 33 010 положительных взаимодействий. Если benchmark query похож/совпадает с train query и соответствующий исторически выбранный item всё ещё находится в benchmark corpus, щначит такого кандидата разумно включать
3. 60.6% текстов запроса имеют всего один уникальный positive item в train, 74.8% - не больше двух, 87.5% - не больше пяти. Но есть маникюр, массаж, педикюр с тысячами items. Поэтому random split по строкам будет плохой валидацией: для популярных запросов train и validation будут почти дублировать друг друга
4. item_category_id=114 - практически железное правило для positives. Из 497 673 положительных строк только 15 относятся не к item_category_id=114. При этом benchmark corpus содержит 47 категорий. Значит большая часть корпуса, вероятно, просто отвлечение

## Анализ поискового контекста
Одинаковый текст запроса может использоваться с разными локациями, категориями и фильтрами. Поэтому проверим количество уникальных поисковых контекстов, определяемых комбинацией:
- `search_query`
- `search_location_id`
- `search_is_delivery_search`
- `search_infm_params_text`
- `search_category`

Это особенно важно для последующего построения локальной валидации.

In [20]:
# Сначала проверим category на benchmark. Перед тем как сказать «выкидываем все items кроме 114», нужно посмотреть, сколько их
benchmark_category_counts = (
    items["item_category_id"]
    .value_counts()
    .sort_values(ascending=False)
)

display(benchmark_category_counts.head(20))

print(
    "\nCategory 114:",
    (items["item_category_id"] == 114).sum()
)

print(
    "Share:",
    (items["item_category_id"] == 114).mean()
)

item_category_id
114    187336
19        357
112       141
40        122
33        103
10         98
20         84
25         82
36         74
111        74
39         72
24         70
83         68
27         57
30         56
97         44
21         36
88         34
82         32
42         29
Name: count, dtype: int64


Category 114: 187336
Share: 0.9900851954421496


In [21]:
# Категории benchmark-запросов
print(queries["search_category"].value_counts())

search_category
114    2230
0       222
Name: count, dtype: int64


In [26]:
# Исследуем исследовать search context
query_cols = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

unique_search_contexts = (
    train[query_cols]
    .drop_duplicates()
)

print("Unique query texts:",
      train["search_query"].nunique())

print("Unique full query contexts:",
      len(unique_search_contexts))

print(
    "Average contexts per query text:",
    len(unique_search_contexts)
    / train["search_query"].nunique()
)

Unique query texts: 74529
Unique full query contexts: 354463
Average contexts per query text: 4.756041272524789


In [27]:
contexts_per_query = (
    unique_search_contexts
    .groupby("search_query")
    .size()
    .sort_values(ascending=False)
)

display(
    contexts_per_query.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

display(contexts_per_query.head(20))

count    74529.000000
mean         4.756041
std         25.811141
min          1.000000
50%          1.000000
75%          2.000000
90%          6.000000
95%         13.000000
99%         64.000000
max       1693.000000
dtype: float64

search_query
массаж                     1693
наращивание ресниц         1553
маникюр                    1499
электрик                   1242
покос травы                1236
сантехник                  1206
установка кондиционеров    1143
эвакуатор                  1078
вывоз мусора               1075
автоэлектрик                956
педикюр                     948
манипулятор                 913
вспашка земли               870
наращивание ногтей          789
грузчики                    779
грузоперевозки              769
заправка кондиционера       757
натяжные потолки            751
поклейка обоев              714
грузоперевозки газель       688
dtype: int64

In [28]:
print(
    "Один и тот же текст запроса встречается в разных локациях:",
    (
        train.groupby("search_query")["search_location_id"]
        .nunique()
        .gt(1)
        .mean()
    )
)

Один и тот же текст запроса встречается в разных локациях: 0.368688698359028


При 74 529 уникальных текстах запросов в train существует 354 463 уникальных поисковых контекста(в среднем около 4,76 контекста на один текст запроса). Кроме того, около 36,9% текстов запросов встречаются более чем в одной локации. Следовательно, один search_query нельзя однозначно считать одним поисковым случаем. При построении валидации необходимо учитывать полный поисковый контекст

## EDA ИТОГИ

- Обучающая выборка содержит 497 673 положительных взаимодействия «поисковый запрос - выбранное объявление»
- В обучающей выборке содержится 74 529 уникальных текстов поисковых запросов и 354 463 уникальных поисковых контекста с учётом локации, категории и фильтров
- 36,99% текстов запросов из benchmark в точности встречаются в обучающей выборке
- 9,59% объявлений из benchmark-корпуса встречаются в обучающей выборке
- Для 60,6% уникальных текстов запросов в обучающей выборке наблюдается только одно релевантное объявление
- Для 87,5% запросов наблюдается не более пяти релевантных объявлений
- В 83,1% положительных взаимодействий `search_location_id` совпадает с `item_location_id`. Поэтому локация является сильным сигналом релевантности, однако использовать её как жёсткий фильтр нельзя, так как примерно в 17% случаев локации не совпадают
- Практически все положительные взаимодействия в обучающей выборке относятся к `item_category_id = 114`
- 99,0% объявлений benchmark-корпуса также относятся к категории 114, поэтому фильтрация по категории существенно не уменьшит пространство поиска
- Один и тот же текст запроса может соответствовать разным поисковым контекстам: примерно 36,9% текстов запросов встречаются более чем в одной локации. Поэтому при построении валидации необходимо учитывать не только `search_query`, но и остальные признаки поискового контекста

Полученные результаты указывают на целесообразность гибридного подхода к кандидатогенерации, объединяющего:

1. лексический поиск по тексту запроса и объявления
2. информацию об исторических взаимодействиях «запрос - объявление»
3. учёт совпадения локации как дополнительного сигнала
4. семантический поиск для случаев, когда запрос и релевантное объявление сформулированы разными словами

Следующим этапом является построение локальной валидации с метрикой `Recall@50`, после чего можно последовательно сравнивать различные методы кандидатогенерации